# Level 4 capstone — Draft pick value model

**Brief.** Predict career Approximate Value (AV) for a drafted player from college production, combine numbers, and draft position. Train on **2000-2020 drafts**, test on **2021-2023 drafts**, then project the **2025 Lions class**.

## Evaluation criteria

1. **Documented features** with a one-line rationale for each. Why this feature?
2. **A real held-out test set** (2021-2023). The model never sees those drafts during training.
3. **Baseline comparison** — at minimum, a 'predict career AV from draft position alone' linear model.
4. **Final report**: model R² on test set, top feature importances, the 2025 Lions class projections (one row per pick).
5. **A one-page writeup** of the model: what it does well, what it misses, what you'd add with more time.

## Data sources

- **Draft picks**: `nfl_data_py.import_draft_picks([2000, ..., 2024])`
- **Draft combine**: `nfl_data_py.import_combine_data([2000, ..., 2024])`
- **Player season stats** (for career AV proxies): `import_seasonal_data` or `import_seasonal_pfr`

Note: nflverse's `import_draft_picks` includes career AV directly via a join to PFR data. Confirm the column name when you pull it.

## Prerequisite

No new Postgres tables — you can pull draft + combine data directly from `nfl_data_py` into pandas. Or, build new `players`, `draft_picks`, `combine` tables in Postgres if you want the Level 5 API to query them later.

## Setup

In [ ]:
import nfl_data_py as nfl
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

LIONS_BLUE = '#0076B6'
TRAIN_YEARS = range(2000, 2021)
TEST_YEARS  = range(2021, 2024)
PROJECT_YEAR = 2025

## Section 1 — Pull the data

Draft picks for 2000-2024. Combine measurables for the same years. Join on player_id (which is the GSIS ID nflverse uses).

In [ ]:
# TODO: pull draft picks (career_av is the target)
# draft = nfl.import_draft_picks(list(range(2000, 2025)))

In [ ]:
# TODO: pull combine data and merge into the draft frame

## Section 2 — Feature engineering

Required features (document why each is in the model):

- `pick_overall` (raw draft position)
- `log_pick`  (`log(pick_overall)` — diminishing returns)
- `position` (one-hot)
- `forty_yd`, `vertical`, `broad_jump`, `bench` (combine metrics; impute or drop NA)
- `age_at_draft` (computed from birthdate − draft date)
- `college_conference` (one-hot SEC, B1G, ACC, ...)

**Document each feature.** A markdown cell after this listing the columns and a one-sentence rationale for each.

In [ ]:
# TODO: build the feature matrix. Handle missing combine values.

*Feature documentation (markdown):*

- `pick_overall` — Single strongest predictor of career outcome; teams aren't random.
- `log_pick` — Diminishing returns: pick 1 vs pick 5 matters more than pick 100 vs pick 104.
- `position` — Different positions have different AV scales (a Pro Bowl WR ≠ a Pro Bowl OL on the AV index).
- `forty_yd` etc — Combine athleticism captures the floor for speed/explosion positions.
- `age_at_draft` — Younger draftees have more time to accumulate AV; older draftees usually peak faster.
- `college_conference` — Crude proxy for level of competition.

## Section 3 — Baseline + model

Train both on 2000-2020. Evaluate on 2021-2023. Don't peek.

In [ ]:
# TODO: split into train (year < 2021) and test (year >= 2021)
# TODO: Baseline 1 — predict mean career_av (DummyRegressor)
# TODO: Baseline 2 — LinearRegression on log_pick only
# TODO: Full model — Pipeline with ColumnTransformer + RandomForestRegressor

In [ ]:
# TODO: print test-set R² and MAE for all three models in a table

## Section 4 — Feature importances + diagnostics

Pull `feature_importances_` from the trained RF. Plot horizontal bar chart. Comment briefly on what surprised you.

In [ ]:
# TODO: feature importance bar chart

## Section 5 — Project the 2025 Lions class

Pull the 2025 Lions picks. Apply the trained pipeline. Output a small table: pick, player, position, predicted career AV, baseline (log-pick) AV, RF AV.

In [ ]:
# TODO: pull 2025 Lions picks
# TODO: build feature matrix in the same shape
# TODO: predict and present

## Section 6 — Writeup

One page in markdown, covering:

- **What the model does well.** Specific evidence from the test-set scores.
- **What it misses.** Players the model called high-AV who busted, or vice versa. Pick 2 examples.
- **What you'd add with more time.** Be specific. "Per-position combine percentiles instead of raw values." "A college production score from PFR." "Hand-graded scouting reports."

---

## Submission checklist

- [ ] All features documented with rationale
- [ ] Train/test split is real — model never saw test years
- [ ] At least two baselines (constant + simple linear)
- [ ] Test R² and MAE reported for all models
- [ ] Feature importance chart with title + axes + source caption
- [ ] 2025 Lions projections table
- [ ] One-page writeup
- [ ] Runs top-to-bottom from a fresh kernel